BERT same word ke liye alag vector bana sakta hai agar context alag ho.

Is code me hum same word 'bank' ko do different contexts me BERT ke through pass karte hain aur dekhte hain ki BERT context ke hisaab se alag embeddings generate karta hai, jabki Word2Vec same embedding deta.

In [7]:
# Run only once in Google Colab

!pip install transformers torch -q

In [8]:
# Tokenizer converts text into tokens
from transformers import BertTokenizer

# BERT model generates contextual embeddings
from transformers import BertModel

# PyTorch backend
import torch

In [9]:
# Tokenizer converts text into tokens
from transformers import BertTokenizer

# BERT model generates contextual embeddings
from transformers import BertModel

# PyTorch backend
import torch

In [10]:
# Load tokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

# Load pretrained BERT model

model = BertModel.from_pretrained(
    "bert-base-uncased"
)

print("BERT Loaded Successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Loaded Successfully


In [11]:
# Same word "bank" used in different contexts

sentence1 = "I went to the bank to deposit money."

sentence2 = "I sat by the river bank watching birds."

In [12]:
for sent in [sentence1, sentence2]:

    print("\nSentence:")
    print(sent)

    # Convert sentence into BERT input format
    inputs = tokenizer(
        sent,
        return_tensors="pt"
    )

    # Disable gradient calculation
    # Faster inference
    with torch.no_grad():

        outputs = model(**inputs)

    # Shape:
    # (batch_size, sequence_length, hidden_size)

    embeddings = outputs.last_hidden_state[0]

    # Convert sentence into tokens

    tokens = tokenizer.tokenize(sent)

    print("\nTokens:")
    print(tokens)

    # Find position of word "bank"

    bank_idx = tokens.index("bank") + 1

    # +1 because BERT adds [CLS] token
    # at the beginning

    bank_embedding = embeddings[bank_idx].numpy()

    print("\nEmbedding of word 'bank'")
    print(bank_embedding[:5])


Sentence:
I went to the bank to deposit money.

Tokens:
['i', 'went', 'to', 'the', 'bank', 'to', 'deposit', 'money', '.']

Embedding of word 'bank'
[ 0.70910335 -0.2590423  -0.01858752 -0.093615    1.2636594 ]

Sentence:
I sat by the river bank watching birds.

Tokens:
['i', 'sat', 'by', 'the', 'river', 'bank', 'watching', 'birds', '.']

Embedding of word 'bank'
[ 0.6383664  -0.39766124  0.14004813 -0.35164765 -0.44892594]


In [13]:
from transformers import (
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [14]:
# Load BERT for classification

clf_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",

    # Example:
    # Positive
    # Negative
    # Neutral

    num_labels=3
)

print("Classification Model Ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classification Model Ready


In [15]:
texts = [
    "I love this movie",
    "This product is terrible",
    "The weather is normal"
]

labels = [
    2,  # Positive
    0,  # Negative
    1   # Neutral
]

In [16]:
encodings = tokenizer(
    texts,

    truncation=True,

    padding=True,

    return_tensors="pt"
)

In [17]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings

        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [18]:
dataset = MyDataset(
    encodings,
    labels
)

In [19]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    logging_steps=1
)

In [20]:
trainer = Trainer(

    model=clf_model,

    args=training_args,

    train_dataset=dataset
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.261608
2,1.020013
3,0.957416
4,1.285197
5,0.842586
6,0.657423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=1.0040405690670013, metrics={'train_runtime': 16.1721, 'train_samples_per_second': 0.557, 'train_steps_per_second': 0.371, 'total_flos': 27750243276.0, 'train_loss': 1.0040405690670013, 'epoch': 3.0})